In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

from utils import resolve_run_dir, load_epoch_latents_and_labels

/home/santripta/miniconda3/envs/llvis/lib/python3.13/site-packages/seaborn/_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.1)
  from scipy.stats import gaussian_kde


In [2]:
MODEL_DATASET = "resnet_cifar"
SPLIT = "trainUval"
EPOCH = 144
TENSOR_TAG = 3
SEED = 241

CANDIDATE_ROOTS = [
    Path("~/data_2/tvcg_multifield_infer").expanduser(),
    Path("~/data_1/tvcg_multifield_infer").expanduser(),
]

In [3]:
run_dir = resolve_run_dir(MODEL_DATASET, CANDIDATE_ROOTS)
X, y = load_epoch_latents_and_labels(run_dir, SPLIT, EPOCH, TENSOR_TAG)

feats_split = pd.read_csv("../../experiment_data/qual_diff/feats_vol10_a3_resnet.csv")
points_split = pd.read_csv("../../experiment_data/qual_diff/points_vol10_a3_resnet.csv")

In [6]:
import pickle as pkl

In [4]:
from sklearn.cluster import AgglomerativeClustering

clust = AgglomerativeClustering()
clust.fit(X)

,n_clusters,2
,metric,'euclidean'
,memory,None
,connectivity,None
,compute_full_tree,'auto'
,linkage,'ward'
,distance_threshold,None
,compute_distances,False


In [7]:
with open("clust_unnorm.pkl", "wb") as f:
    pkl.dump(clust, f)

In [8]:
with open("clust_unnorm.pkl", "rb") as f:
    clust = pkl.load(f)

In [28]:
import networkx as nx

N = X.shape[0]
T = nx.DiGraph()

for (i, j) in clust.children_:
    T.add_node(i)
    T.add_node(j)

for i in range(N):
    if i not in T.nodes:
        T.add_node(i)
        print(f"Added leaf node {i} to tree")

for i, (c1, c2) in enumerate(clust.children_):
    id = i + N
    T.add_edge(id, c1)
    T.add_edge(id, c2)

In [79]:
feats_interest = [(45,), (13,), (45, 13), (4,), (17,), (4, 17)]
ids = [feats_split[feats_split["Feature ID"].isin(fs)]["Feature ID"].unique() for fs in feats_interest]
points = [points_split[points_split["Feature ID"].isin(id_set)]["Data Index"].unique() for id_set in ids]

[len(p) for p in points]

[63, 52, 115, 25, 6, 31]

In [69]:
rand_points = np.random.randint(0, N, (10, 2))

In [81]:
def subcomp_size(node):
    desc = nx.descendants(T, node)
    leaves = [n for n in desc if n < N]
    return len(leaves)

def compute_subcomp_size(points):
    lcas = [nx.lowest_common_ancestor(T, points[0], points[1])]
    for p in points[2:]:
        lca = nx.lowest_common_ancestor(T, lcas[-1], p)
        lcas.append(lca)

    return subcomp_size(lcas[-1])

In [82]:
feat_sizes = [compute_subcomp_size(p) for p in points]
feat_sizes

[19965, 27700, 27700, 60000, 60000, 60000]

In [74]:
rand_sizes = [compute_subcomp_size(points) for points in rand_points]

In [75]:
rand_sizes

[23673, 27700, 23673, 36327, 60000, 11365, 32317, 32317, 2059, 15323]